# Prototype 02 — Run POSS-LOGIC-LM from a Beam Database

This notebook shows how to run the **POSS-LOGIC-LM aggregation pipeline** from an existing beam-search database.

It does **not** regenerate formalizations.  
It assumes you already have a database JSON file such as:

```text
prontoqa_gemma.json
proofwriter_gemma.json
folio_gemma.json
```

Default configuration:
- Dataset: **PrOntoQA**
- Solver: **SWI-Prolog**
- Input database: `prontoqa_gemma.json`
- Output results: `pbs_prontoqa_gemma_results.json`

## Hugging Face token

This notebook does not need a Hugging Face token if it only reads an existing database.  
You need `HF_TOKEN` only when generating new formalizations with a model.

In Kaggle:
1. Open the notebook.
2. Go to **Add-ons → Secrets**.
3. Add a secret named `HF_TOKEN`.
4. Paste your Hugging Face token.

## How to change the input model

To evaluate another model, change only the input database and output file:

```python
INPUT_DATABASE = "prontoqa_qwen.json"
OUTPUT_RESULTS = "pbs_prontoqa_qwen_results.json"
```

or:

```python
INPUT_DATABASE = "prontoqa_llama3.json"
OUTPUT_RESULTS = "pbs_prontoqa_llama3_results.json"
```

In [ ]:
!apt-get install -y swi-prolog -q
!pip install -q tqdm

## 1. Configuration

In [ ]:
import os
import re
import json
import math
import glob
import shutil
import subprocess
import tempfile
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

# -----------------------------
# User configuration
# -----------------------------
DATASET_NAME = "PrOntoQA"
INPUT_DATABASE = "prontoqa_gemma.json"
OUTPUT_RESULTS = Path("pbs_prontoqa_gemma_results.json")

LABELS = ["True", "False"]  # PrOntoQA is binary
TRANSFORM = "T1_Klir"
BEAM_K = 5

print("Dataset:", DATASET_NAME)
print("Input database:", INPUT_DATABASE)
print("Output results:", OUTPUT_RESULTS)

## 2. Locate the database file

On Kaggle, upload the database as a dataset or attach it as notebook input.  
This cell searches in `/kaggle/input/` and in the current working directory.

In [ ]:
def find_database(filename: str):
    candidates = []
    candidates.extend(glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
    candidates.extend(glob.glob(f"./{filename}", recursive=True))

    if not candidates:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach it as a Kaggle input dataset or upload it to the working directory."
        )

    return candidates[0]

db_path = find_database(INPUT_DATABASE)

with open(db_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Database path:", db_path)
print("Examples:", len(data))
print("First keys:", list(data[0].keys()))

## 3. Probability-to-possibility transformation

We apply a stable softmax over the beam scores, then max-normalize the values:

```text
pi(F_i) = q_i / max_j q_j
```

In [ ]:
def stable_softmax(scores):
    clean_scores = []
    for s in scores:
        if s is None:
            clean_scores.append(-1e9)
        else:
            clean_scores.append(float(s))

    max_s = max(clean_scores)
    exps = [math.exp(s - max_s) for s in clean_scores]
    total = sum(exps)

    if total == 0:
        return [1.0 / len(clean_scores)] * len(clean_scores)

    return [e / total for e in exps]

def scores_to_possibilities(log_scores):
    q = stable_softmax(log_scores)
    max_q = max(q)
    if max_q == 0:
        return [0.0 for _ in q]
    return [x / max_q for x in q]

## 4. Minimal Prolog utilities

This prototype assumes that the generated formalizations are already close to Prolog-style Horn clauses.

If your database format is different, adapt `extract_program_and_query()`.

In [ ]:
def normalize_label(label):
    s = str(label).strip().lower()
    if s in {"true", "proved", "yes"}:
        return "True"
    if s in {"false", "disproved", "no"}:
        return "False"
    return "Unknown"

def sanitize_program(program: str):
    # Keep this simple in the prototype.
    # The full project code contains more robust normalization.
    lines = []
    for line in str(program).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("%") or line.lower().startswith(("predicates", "facts", "rules", "query")):
            continue
        if line.startswith("- "):
            line = line[2:].strip()
        if line and not line.endswith("."):
            line += "."
        lines.append(line)
    return "\n".join(lines)

def extract_query_from_formalization(text: str):
    lines = str(text).splitlines()
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("query"):
            # Examples: QUERY: sour(max). or QUERY:
            after = line.split(":", 1)[-1].strip() if ":" in line else ""
            if after:
                return after.rstrip(".") + "."
            if i + 1 < len(lines):
                return lines[i + 1].strip().lstrip("- ").rstrip(".") + "."
    return None

def extract_program_and_query(row, formalization):
    query = row.get("query")
    if not query:
        query = extract_query_from_formalization(formalization)

    if not query:
        # Fallback for PrOntoQA if query is unavailable.
        # For serious experiments, the database should store the parsed query.
        conclusion = row.get("conclusion", "")
        query = conclusion.lower().replace(" is ", "_").replace(" ", "_").replace(".", "")
        query = query + "."

    program = sanitize_program(formalization)
    return program, query

def run_prolog(program: str, query: str, timeout: int = 10):
    query = query.strip().rstrip(".")
    prolog_code = f"""
:- initialization(main).

{program}

main :-
    (call(({query})) ->
        writeln('RESULT=true')
    ;
        writeln('RESULT=false')
    ),
    halt.
"""

    with tempfile.NamedTemporaryFile("w", suffix=".pl", delete=False, encoding="utf-8") as f:
        f.write(prolog_code)
        path = f.name

    try:
        proc = subprocess.run(
            ["swipl", "-q", "-f", path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout,
        )
        out = proc.stdout.strip().lower()
        success = proc.returncode == 0

        if "result=true" in out:
            return "True", success
        return "False", success

    except Exception:
        return "Unknown", False

    finally:
        try:
            os.remove(path)
        except OSError:
            pass

## 5. POSS-LOGIC-LM aggregation

In [ ]:
def aggregate_solver_outputs(labels, possibilities, label_space=LABELS):
    # Possibility of each label: max possibility among candidates supporting it.
    pi = {label: 0.0 for label in label_space}
    weighted = {label: 0.0 for label in label_space}

    for label, poss in zip(labels, possibilities):
        if label not in pi:
            continue
        pi[label] = max(pi[label], poss)
        weighted[label] += poss

    max_pi = max(pi.values())
    candidates = [label for label, value in pi.items() if value == max_pi]

    if len(candidates) == 1:
        pred = candidates[0]
    else:
        pred = max(candidates, key=lambda label: weighted[label])

    necessity = 1.0 - max([pi[l] for l in label_space if l != pred] or [0.0])

    return pred, pi, weighted, necessity

## 6. Evaluate one example

In [ ]:
def evaluate_row(row):
    formalisations = row.get("formalisations", [])[:BEAM_K]
    log_scores = row.get("log_scores", [])[:BEAM_K]

    possibilities = scores_to_possibilities(log_scores)
    solver_labels = []
    candidate_success = []

    for formalization in formalisations:
        program, query = extract_program_and_query(row, formalization)
        label, success = run_prolog(program, query)
        solver_labels.append(label)
        candidate_success.append(success)

    pred, pi, weighted, necessity = aggregate_solver_outputs(solver_labels, possibilities, LABELS)

    gold = normalize_label(row.get("label", row.get("gold", "Unknown")))

    return {
        "id": row.get("id"),
        "gold": gold,
        "pred": pred,
        "correct": pred == gold,
        "solver_outputs": solver_labels,
        "candidate_success": candidate_success,
        "possibilities": possibilities,
        "pi": pi,
        "weighted": weighted,
        "necessity": necessity,
    }

# Quick test
for i in range(min(3, len(data))):
    result = evaluate_row(data[i])
    print(result["id"], "gold=", result["gold"], "pred=", result["pred"], "solver=", result["solver_outputs"])

## 7. Run evaluation

In [ ]:
results = []
correct = 0
executable = 0

for row in tqdm(data):
    result = evaluate_row(row)
    results.append(result)

    correct += int(result["correct"])
    executable += int(any(result["candidate_success"]))

total = len(results)
accuracy = round(100 * correct / total, 2) if total else 0.0
exe_rate = round(100 * executable / total, 2) if total else 0.0

summary = {
    "dataset": DATASET_NAME.lower(),
    "method": "POSS-LOGIC-LM",
    "transform": TRANSFORM,
    "stats": {
        "total": total,
        "correct": correct,
        "accuracy": accuracy,
        "exe_rate": exe_rate,
    },
    "results": results,
}

with open(OUTPUT_RESULTS, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary["stats"], indent=2))
print("Saved:", OUTPUT_RESULTS)

## 8. Inspect errors

In [ ]:
wrong = [r for r in results if not r["correct"]]
print("Wrong examples:", len(wrong))

for r in wrong[:5]:
    print(r["id"], "gold=", r["gold"], "pred=", r["pred"], "solver=", r["solver_outputs"], "N=", round(r["necessity"], 3))

## 9. Notes

This is a compact prototype for repository reproducibility.

The full experimental notebooks may include:
- stronger Prolog normalization;
- dataset-specific query extraction;
- FOLIO + Prover9 support;
- LogicLM self-refinement;
- resume mode;
- detailed error analysis.